# 🛍️ Chatbot E-Commerce avec LLM — Assistant Conversationnel Multimodal
## Master NLP Avancé — ESGIS Bénin

**Groupe :** ALAMISSI Ibikitan Alice Marlise · BOCO Cossi Hospice Francky · GANDONOU Schalom Miguel · LANGANFIN Renaud Rogelio

---

### Architecture du pipeline
```
Voix client (.m4a)                       Requête texte
      │                                        │
      ▼                                        │
 [Whisper ASR]  ──────────────────────────────▶│
 (Phase 3)                                     ▼
                               [Recherche sémantique]
                               TF-IDF | MiniLM Neural
                                   (Phase 2)
                                        │
                               Top-3 produits
                                        │
                                        ▼
                               [Génération LLM]
                           Gemini / OpenAI / Mistral
                                   (Phase 4)
                                        │
                               Réponse naturelle
```

### Table des matières
- **[SETUP]** Installation & imports globaux
- **[PHASE 1]** Chargement catalogue + ground truth
- **[PHASE 2]** Moteur de recherche sémantique (TF-IDF vs Neural)
- **[PHASE 3]** ASR Whisper (transcription + WER + impact)
- **[PHASE 4]** Génération LLM (prompting + évaluation)
- **[BONUS]**  Pipeline Gradio interactif (démo live)

---
# ⚙️ SETUP — Installation & Configuration

> **Avant tout :** `Exécution → Modifier le type d'exécution → GPU T4`  
> **Une seule exécution suffit** — toutes les phases partagent les mêmes objets en mémoire.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# INSTALLATION DE TOUTES LES DÉPENDANCES DU PROJET
#
# sentence-transformers : embeddings neuronaux (MiniLM multilingue)
# scikit-learn          : TF-IDF, similarité cosinus, t-SNE
# openai-whisper        : reconnaissance vocale ASR
# jiwer                 : calcul WER/CER pour évaluation ASR
# librosa / soundfile   : chargement et manipulation des fichiers audio
# google-generativeai   : API Gemini (LLM gratuit) pour la Phase 4
# umap-learn            : visualisation alternative au t-SNE
# ══════════════════════════════════════════════════════════════════════════════
print("📦 Installation des dépendances (2-3 minutes)...")

!pip install -q openai  # SDK OpenAI — compatible NVIDIA NIM
!pip install -q sentence-transformers scikit-learn umap-learn
!pip install -q openai-whisper jiwer librosa soundfile
!pip install -q google-generativeai
!pip install -q matplotlib seaborn pandas numpy tqdm
!apt-get install -qq ffmpeg   # nécessaire pour que Whisper lise les .m4a

print("\n✅ Toutes les dépendances sont installées.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# IMPORTS GLOBAUX — utilisés par toutes les phases
# ══════════════════════════════════════════════════════════════════════════════
import os, time, json, re, difflib, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
warnings.filterwarnings('ignore')

# NLP / ML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

# ASR
import whisper
import torch
from jiwer import wer, cer
from jiwer import transforms as tr

# LLM
from openai import OpenAI  # SDK OpenAI compatible avec NVIDIA NIM

np.random.seed(42)

print(f"✅ Imports OK | GPU : {torch.cuda.is_available()} ", end="")
if torch.cuda.is_available():
    print(f"({torch.cuda.get_device_name(0)})")
else:
    print("\n⚠️  Pas de GPU — Whisper large sera très lent. Activez le GPU dans les paramètres.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# UPLOAD DES FICHIERS
#
# Option A (recommandée) : uploader via le panneau gauche de Colab
#   → icône 📁 → bouton « Importer un fichier »
#   → Fichiers nécessaires :
#        africouleur_products.csv
#        africouleur_ground_truth.csv
#        dossier audio/ avec les fichiers .m4a
#
# Option B : Google Drive
# ──────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# CATALOGUE_PATH = '/content/drive/MyDrive/NLP_Projet/africouleur_products.csv'
# GT_PATH        = '/content/drive/MyDrive/NLP_Projet/africouleur_ground_truth.csv'
# ══════════════════════════════════════════════════════════════════════════════

CATALOGUE_PATH = 'africouleur_products.csv'
GT_PATH        = 'africouleur_ground_truth.csv'
AUDIO_DIR      = 'audio'                        # dossier contenant les .m4a

# Clé API NVIDIA NIM (gratuite avec crédits)
# Inscription : https://build.nvidia.com  →  Get API Key
NVIDIA_API_KEY  = 'VOTRE_CLE_NVIDIA_ICI'
NVIDIA_BASE_URL = 'https://integrate.api.nvidia.com/v1'
# Clé API NVIDIA NIM (gratuite avec crédits)
# Inscription : https://build.nvidia.com  →  Get API Key
NVIDIA_API_KEY  = 'VOTRE_CLE_NVIDIA_ICI'
NVIDIA_BASE_URL = 'https://integrate.api.nvidia.com/v1'

os.makedirs(AUDIO_DIR, exist_ok=True)
print(f"📂 Catalogue   : {CATALOGUE_PATH}")
print(f"📂 Ground truth: {GT_PATH}")
print(f"📂 Audio dir   : {AUDIO_DIR}/")
# Clé API NVIDIA NIM (gratuite avec crédits)
# Inscription : https://build.nvidia.com  →  Get API Key
NVIDIA_API_KEY  = 'VOTRE_CLE_NVIDIA_ICI'
NVIDIA_BASE_URL = 'https://integrate.api.nvidia.com/v1'


---
# 📦 PHASE 1 — Catalogue & Corpus de Requêtes
*(25% de la note)*

Le catalogue a été scrapé sur **africouleur.com** (~300 produits réels).  
50 requêtes clients ont été rédigées et annotées manuellement.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 1.1 CHARGEMENT DU CATALOGUE
# Colonnes : name, short_description, description, category, sub_category,
#            price_fcfa, tags, url_product
# ──────────────────────────────────────────────────────────────────────────────
df = pd.read_csv(CATALOGUE_PATH).reset_index(drop=True)
df['id'] = df.index + 1   # ID 1-based pour correspondre aux annotations GT

print(f"📦 Catalogue chargé : {len(df)} produits")
print(f"   Colonnes : {list(df.columns[:8])}...")
print()
print("📊 Répartition par catégorie :")
print(df['category'].value_counts().to_string())
print()
print("--- Aperçu (3 premiers produits) ---")
df[['id','name','category','price_fcfa']].head(3)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 1.2 CHARGEMENT DU GROUND TRUTH
# 50 requêtes annotées : 3 produits pertinents par requête (annotation manuelle)
# Types : Recherche directe / Recherche par besoin / Question sur un produit
# ──────────────────────────────────────────────────────────────────────────────
gt = pd.read_csv(GT_PATH)

def parse_relevant_ids(idx_str):
    """Convertit '723, 728, 726' → [723, 728, 726]"""
    try:
        return [int(x.strip()) for x in str(idx_str).split(',')]
    except:
        return []

gt['relevant_ids'] = gt['Index'].apply(parse_relevant_ids)

# Requêtes avec fichier audio
gt_audio = gt[
    gt['Audio_Français'].notna() & (gt['Audio_Français'].str.strip() != '')
].copy().reset_index(drop=True)

print(f"📋 Ground truth : {len(gt)} requêtes")
print(f"   Types : {gt['Type de requête'].value_counts().to_dict()}")
print(f"   Requêtes avec audio : {len(gt_audio)}")
print()
gt[['ID','Type de requête','Requête client','relevant_ids']].head(5)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 1.3 CONSTRUCTION DU TEXTE COMBINÉ PAR PRODUIT
# Concaténation : nom + description courte + description longue[:400] + tags + catégorie
# Ce texte unique est la base des deux méthodes d'embedding (Phase 2).
# ──────────────────────────────────────────────────────────────────────────────
def build_text_combined(row):
    parts = []
    for champ in ['name', 'short_description']:
        v = str(row.get(champ, '')).strip()
        if v and v != 'nan': parts.append(v)
    ld = str(row.get('description', '')).strip()[:400]
    if ld and ld != 'nan': parts.append(ld)
    tags = str(row.get('tags', '')).replace('|', ' ').strip()
    if tags and tags != 'nan': parts.append(tags)
    for champ in ['category', 'sub_category']:
        v = str(row.get(champ, ''))
        if v and v != 'nan': parts.append(v)
    return ' '.join(parts)

df['text_combined'] = df.apply(build_text_combined, axis=1)
print(f"✅ Textes combinés générés")
print(f"   Longueur moyenne : {df['text_combined'].str.len().mean():.0f} chars")
print()
print("Exemple produit #1 :")
print(df.iloc[0]['text_combined'][:300])

---
# 🔍 PHASE 2 — Moteur de Recherche Sémantique
*(30% de la note)*

Deux méthodes comparées sur les 50 requêtes du ground truth.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 2.1 MÉTHODE 1 : TF-IDF (BASELINE)
#
# Représentation creuse basée sur la fréquence pondérée des termes.
# Paramètres :
#   ngram_range=(1,2) : unigrammes + bigrammes pour capturer "tissu bazin", etc.
#   min_df=1          : conserver les termes rares (vocabulaire local africain)
#   sublinear_tf=True : log(tf+1) pour atténuer les termes très fréquents
# ──────────────────────────────────────────────────────────────────────────────
print("⚙️  Construction TF-IDF...")
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2), max_features=20000,
    min_df=1, sublinear_tf=True, strip_accents=None
)
tfidf_matrix = tfidf_vectorizer.fit_transform(df['text_combined'])
print(f"   ✅ Matrice TF-IDF : {tfidf_matrix.shape}")

def recherche_tfidf(requete: str, k: int = 5) -> list:
    """
    Recherche TF-IDF : encode la requête et calcule la similarité cosinus
    avec tous les produits. Retourne les k meilleurs (product_id, score).
    """
    q_vec  = tfidf_vectorizer.transform([requete])
    scores = cosine_similarity(q_vec, tfidf_matrix).flatten()
    top_k  = np.argsort(scores)[::-1][:k]
    return [(int(df.iloc[i]['id']), float(scores[i])) for i in top_k]

# Test rapide
test_r = recherche_tfidf("Je cherche un pagne wax", k=3)
print(f"   Test → {[(df[df.id==p]['name'].values[0][:40], f'{s:.3f}') for p,s in test_r]}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 2.2 MÉTHODE 2 : EMBEDDINGS NEURONAUX — MiniLM Multilingue
#
# Modèle : paraphrase-multilingual-MiniLM-L12-v2
# → 12 couches Transformer, 384 dimensions, 50+ langues
# → Capture la sémantique : "pagne wax" ≈ "tissu africain" dans l'espace vectoriel
# → normalize_embeddings=True : norme=1 → cosinus = produit scalaire (rapide)
#
# ⏱️ ~2-3 min pour encoder ~300 produits
# ──────────────────────────────────────────────────────────────────────────────
print("📥 Chargement MiniLM multilingue...")
model_emb = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
print(f"   Dimension : {model_emb.get_sentence_embedding_dimension()}D")

print("\n⚙️  Encodage du catalogue...")
corpus_embeddings = model_emb.encode(
    df['text_combined'].tolist(),
    batch_size=64, show_progress_bar=True, normalize_embeddings=True
)
print(f"\n   ✅ Shape : {corpus_embeddings.shape}")

def recherche_semantique(requete: str, k: int = 5) -> list:
    """
    Recherche neuronale : encode la requête avec MiniLM puis calcule la
    similarité cosinus via produit matriciel (vecteurs normalisés).
    Retourne les k meilleurs (product_id, cosine_score).
    """
    q_emb  = model_emb.encode([requete], normalize_embeddings=True)
    scores = (corpus_embeddings @ q_emb.T).flatten()
    top_k  = np.argsort(scores)[::-1][:k]
    return [(int(df.iloc[i]['id']), float(scores[i])) for i in top_k]

# Test rapide
test_r = recherche_semantique("Je cherche un pagne wax", k=3)
print(f"   Test → {[(df[df.id==p]['name'].values[0][:40], f'{s:.3f}') for p,s in test_r]}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 2.3 MÉTRIQUES D'ÉVALUATION
#
# Precision@K  : proportion de produits pertinents dans les K premiers résultats
# MRR          : 1 / rang du 1er résultat pertinent (1.0 = toujours en #1)
# Hit Rate@K   : 1 si au moins un produit pertinent dans le top-K
# ──────────────────────────────────────────────────────────────────────────────
def precision_at_k(retrieved, relevant, k=5):
    ids = [r[0] for r in retrieved[:k]]
    return sum(1 for i in ids if i in relevant) / k

def mrr(retrieved, relevant):
    for rang, (rid, _) in enumerate(retrieved, 1):
        if rid in relevant: return 1.0 / rang
    return 0.0

def hit_rate_at_k(retrieved, relevant, k=5):
    ids = [r[0] for r in retrieved[:k]]
    return 1.0 if any(i in relevant for i in ids) else 0.0

print("✅ Métriques définies : precision_at_k, mrr, hit_rate_at_k")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 2.4 ÉVALUATION COMPARATIVE SUR LES 50 REQUÊTES
# ──────────────────────────────────────────────────────────────────────────────
print("⚙️  Évaluation des 2 moteurs sur 50 requêtes...")
eval_rows = []

for _, row in tqdm(gt.iterrows(), total=len(gt)):
    query    = row['Requête client']
    relevant = row['relevant_ids']

    res_tf = recherche_tfidf(query, k=5)
    res_nn = recherche_semantique(query, k=5)

    eval_rows.append({
        'query_id'    : int(row['ID']),
        'type'        : row['Type de requête'],
        'query'       : query,
        'relevant_ids': relevant,
        'p5_tfidf'    : precision_at_k(res_tf, relevant),
        'mrr_tfidf'   : mrr(res_tf, relevant),
        'hr5_tfidf'   : hit_rate_at_k(res_tf, relevant),
        'top5_tfidf'  : [r[0] for r in res_tf],
        'p5_neural'   : precision_at_k(res_nn, relevant),
        'mrr_neural'  : mrr(res_nn, relevant),
        'hr5_neural'  : hit_rate_at_k(res_nn, relevant),
        'top5_neural' : [r[0] for r in res_nn],
    })

eval_df = pd.DataFrame(eval_rows)

# Tableau comparatif
print("\n" + "═"*60)
print("  TABLEAU COMPARATIF — TF-IDF vs Embeddings Neuronaux")
print("═"*60)
for label, p, m, h in [
    ('TF-IDF',          eval_df.p5_tfidf.mean(),  eval_df.mrr_tfidf.mean(),  eval_df.hr5_tfidf.mean()),
    ('Neural (MiniLM)', eval_df.p5_neural.mean(), eval_df.mrr_neural.mean(), eval_df.hr5_neural.mean()),
]:
    print(f"  {label:<22} P@5={p:.4f}  MRR={m:.4f}  HR@5={h:.4f}")
print("═"*60)

# Analyse par type de requête
print()
for rtype in eval_df['type'].unique():
    sub = eval_df[eval_df['type'] == rtype]
    print(f"  [{rtype}] ({len(sub)} req.) → TF-IDF P@5={sub.p5_tfidf.mean():.3f} | Neural P@5={sub.p5_neural.mean():.3f}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 2.5 VISUALISATIONS PHASE 2
# ──────────────────────────────────────────────────────────────────────────────

# ── Graphe 1 : Barres comparatives globales ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barres globales
ax = axes[0]
metrics = ['Precision@5', 'MRR', 'Hit Rate@5']
x = np.arange(3)
ax.bar(x-0.18, [eval_df.p5_tfidf.mean(), eval_df.mrr_tfidf.mean(), eval_df.hr5_tfidf.mean()],
       0.35, label='TF-IDF', color='#E07B54', edgecolor='white')
ax.bar(x+0.18, [eval_df.p5_neural.mean(), eval_df.mrr_neural.mean(), eval_df.hr5_neural.mean()],
       0.35, label='Neural (MiniLM)', color='#3A86FF', edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylim(0, 1.15); ax.legend(fontsize=10)
ax.set_title('TF-IDF vs Neural — Métriques globales', fontweight='bold')
ax.spines[['top','right']].set_visible(False)

# P@5 par requête
ax = axes[1]
type_colors = {'Recherche directe':'#4CAF50','Recherche par besoin':'#FF9800','Question sur un produit':'#9C27B0'}
bar_colors  = [type_colors.get(t,'#888') for t in eval_df['type']]
ax.bar(range(len(eval_df)), eval_df['p5_neural'], color=bar_colors, alpha=0.8)
ax.axhline(eval_df.p5_neural.mean(), color='black', linestyle='--', lw=1.5,
           label=f'Moy={eval_df.p5_neural.mean():.3f}')
ax.set_title('P@5 Neural par requête', fontweight='bold')
ax.set_ylim(0, 1.1); ax.legend(fontsize=9)
patches = [mpatches.Patch(color=c, label=t) for t,c in type_colors.items()]
ax.legend(handles=patches, fontsize=8)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('phase2_comparaison.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Graphe 2 : t-SNE ──────────────────────────────────────────────────────────
print("⚙️  Calcul t-SNE (1-2 min)...")
query_embs = model_emb.encode(gt['Requête client'].tolist(), normalize_embeddings=True)
all_embs   = np.vstack([corpus_embeddings, query_embs])
tsne_proj  = TSNE(n_components=2, perplexity=30, random_state=42, init='pca').fit_transform(all_embs)
prod_proj  = tsne_proj[:len(df)]
qr_proj    = tsne_proj[len(df):]

cats       = df['category'].fillna('Autre').tolist()
cat_uniq   = sorted(set(cats))
cmap       = plt.cm.get_cmap('tab10', len(cat_uniq))
c2col      = {c: cmap(i) for i,c in enumerate(cat_uniq)}
pcolors    = [c2col[c] for c in cats]

fig, ax = plt.subplots(figsize=(14, 10))
ax.scatter(prod_proj[:,0], prod_proj[:,1], c=pcolors, alpha=0.5, s=22, zorder=2)
ax.scatter(qr_proj[:,0],   qr_proj[:,1],   c='black', marker='*', s=180, zorder=5,
           edgecolors='gold', linewidths=0.7, label='Requêtes clients')
patches = [mpatches.Patch(color=c2col[c], label=c) for c in cat_uniq]
patches.append(plt.Line2D([0],[0], marker='*', color='black', markerfacecolor='black',
                           markersize=10, linestyle='None', label='Requêtes clients'))
ax.legend(handles=patches, fontsize=8, ncol=2, loc='upper left')
ax.set_title('Espace d\'embeddings MiniLM — t-SNE 2D\nProduits Africouleur + Requêtes clients',
             fontsize=13, fontweight='bold')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('phase2_tsne.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Sauvegardés : phase2_comparaison.png, phase2_tsne.png")

---
# 🎙️ PHASE 3 — Intégration ASR (Whisper)
*(20% de la note)*

**Instructions audio :** Uploader les fichiers `.m4a` dans le dossier `audio/` via le panneau 📁 de Colab.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 3.1 CHARGEMENT DES MODÈLES WHISPER
#
# Whisper small    : 244M params, rapide, précision correcte
# Whisper large-v3 : 1550M params, lent, meilleure précision sur les accents
# Les deux sont chargés localement (pas d'API, pas de coût).
# ──────────────────────────────────────────────────────────────────────────────
print("📥 Chargement Whisper small...")
t0 = time.time()
whisper_small = whisper.load_model("small")
print(f"   ✅ small chargé en {time.time()-t0:.1f}s")

print("\n📥 Chargement Whisper large-v3 (~3 GB, patient une fois)...")
t0 = time.time()
whisper_large = whisper.load_model("large-v3")
print(f"   ✅ large-v3 chargé en {time.time()-t0:.1f}s")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 3.2 VÉRIFICATION DES FICHIERS AUDIO
# ──────────────────────────────────────────────────────────────────────────────
audio_data = []
print("🔍 Fichiers audio attendus :")

for _, row in gt_audio.iterrows():
    path_raw  = row['Audio_Français'].strip()
    path_norm = path_raw.lstrip('./')
    found     = path_raw if os.path.exists(path_raw) else (path_norm if os.path.exists(path_norm) else None)
    status    = "✅" if found else "❌ MANQUANT"
    print(f"  {status} [{row['ID']:>2}] {os.path.basename(path_raw)}")
    if found:
        audio_data.append({'id': int(row['ID']), 'chemin': found, 'ref': row['Requête client']})

print(f"\n🎯 {len(audio_data)} fichier(s) audio disponible(s) pour la transcription.")
if len(audio_data) == 0:
    print("\n⚠️  Uploader les .m4a dans le dossier 'audio/' via le panneau gauche de Colab.")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 3.3 TRANSCRIPTION AVEC LES 2 MODÈLES WHISPER
#
# Options :
#   language='fr'   : force le français (évite la confusion avec l'anglais)
#   temperature=0   : mode déterministe pour la reproductibilité
#   fp16            : précision demi-flottant si GPU (2x plus rapide)
# ──────────────────────────────────────────────────────────────────────────────
normalisation_wer = tr.Compose([
    tr.ToLowerCase(), tr.RemovePunctuation(),
    tr.RemoveMultipleSpaces(), tr.Strip()
])

def transcrire(modele, chemin, langue='fr'):
    """Transcrit un fichier audio avec Whisper. Retourne texte + temps."""
    t0 = time.time()
    r  = modele.transcribe(chemin, language=langue, task='transcribe',
                           temperature=0, fp16=torch.cuda.is_available())
    return {'text': r['text'].strip(), 'time_s': time.time()-t0}

def calc_wer(ref, hyp):
    rn = normalisation_wer(ref)
    hn = normalisation_wer(hyp)
    return wer(rn, hn) if rn else 0.0

transcriptions = []
print(f"🎙️  Transcription de {len(audio_data)} fichier(s)...\n")

for item in tqdm(audio_data, desc="Whisper"):
    rs = transcrire(whisper_small, item['chemin'])
    rl = transcrire(whisper_large, item['chemin'])
    transcriptions.append({
        'query_id'     : item['id'],
        'reference'    : item['ref'],
        'whisper_small': rs['text'],
        'whisper_large': rl['text'],
        'wer_small'    : calc_wer(item['ref'], rs['text']),
        'wer_large'    : calc_wer(item['ref'], rl['text']),
        'cer_small'    : cer(normalisation_wer(item['ref']), normalisation_wer(rs['text'])),
        'cer_large'    : cer(normalisation_wer(item['ref']), normalisation_wer(rl['text'])),
        'time_small'   : rs['time_s'],
        'time_large'   : rl['time_s'],
    })
    print(f"[{item['id']:>2}] REF  : {item['ref']}")
    print(f"       SMALL: {rs['text']}  (WER={transcriptions[-1]['wer_small']:.3f})")
    print(f"       LARGE: {rl['text']}  (WER={transcriptions[-1]['wer_large']:.3f})")
    print()

trans_df = pd.DataFrame(transcriptions)
print("═"*55)
print(f"  WER moyen — Whisper small   : {trans_df.wer_small.mean():.4f}")
print(f"  WER moyen — Whisper large-v3: {trans_df.wer_large.mean():.4f}")
print("═"*55)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 3.4 IMPACT ASR SUR LA RECHERCHE
#
# Pour chaque requête audio, on compare P@5 avec :
#   - Le texte de référence (sans erreur)
#   - La transcription Whisper small
#   - La transcription Whisper large-v3
# Dégradation = P@5(ref) − P@5(transcription)
# ──────────────────────────────────────────────────────────────────────────────
impact_rows = []

for _, row in trans_df.iterrows():
    gt_row   = gt[gt['ID'] == row['query_id']].iloc[0]
    relevant = gt_row['relevant_ids']

    r_ref   = recherche_semantique(row['reference'],     k=5)
    r_small = recherche_semantique(row['whisper_small'], k=5)
    r_large = recherche_semantique(row['whisper_large'], k=5)

    p5_ref   = precision_at_k(r_ref,   relevant)
    p5_small = precision_at_k(r_small, relevant)
    p5_large = precision_at_k(r_large, relevant)

    impact_rows.append({
        'query_id'       : int(row['query_id']),
        'reference'      : row['reference'],
        'whisper_small'  : row['whisper_small'],
        'whisper_large'  : row['whisper_large'],
        'wer_small'      : row['wer_small'],
        'wer_large'      : row['wer_large'],
        'p5_ref'         : p5_ref,
        'p5_small'       : p5_small,
        'p5_large'       : p5_large,
        'degr_small'     : p5_ref - p5_small,   # dégradation due à Whisper small
        'degr_large'     : p5_ref - p5_large,   # dégradation due à Whisper large
        'cassee_small'   : int(hit_rate_at_k(r_small, relevant) == 0),
        'cassee_large'   : int(hit_rate_at_k(r_large, relevant) == 0),
    })

impact_df = pd.DataFrame(impact_rows)

print("═"*65)
print("  IMPACT ASR SUR LA RECHERCHE (Neural MiniLM)")
print("═"*65)
print(f"  Référence    → P@5={impact_df.p5_ref.mean():.4f}")
print(f"  Whisper small→ P@5={impact_df.p5_small.mean():.4f}  Dégr.={impact_df.degr_small.mean():+.4f}  Cassées={impact_df.cassee_small.sum()}")
print(f"  Whisper large→ P@5={impact_df.p5_large.mean():.4f}  Dégr.={impact_df.degr_large.mean():+.4f}  Cassées={impact_df.cassee_large.sum()}")
print("═"*65)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 3.5 ANALYSE DÉTAILLÉE DES ERREURS DE TRANSCRIPTION
#
# Catégories d'erreurs (selon le sujet) :
#   A) Mot-clé critique manqué (wax, bazin, bogolan...)
#   B) Vocabulaire local africain absent du modèle ASR
#   C) Code-switching français / langue locale (fon, yoruba)
#   D) Accent béninois / prononciation locale
# ──────────────────────────────────────────────────────────────────────────────
MOTS_CLES_CRITIQUES = {
    'wax','bazin','bogolan','pagne','tissu','robe','dashiki','boubou',
    'collier','bracelet','boucles','masque','sculpture','awalé','karité',
    'cauris','touareg','peul','ankara','mariage','cérémonie','cadeau',
    'sac','bijou','bijoux','rideau','calicot'
}

def analyser_erreurs(reference, hypothese):
    """Identifie les erreurs mot-à-mot entre référence et transcription."""
    ref_mots = normalisation_wer(reference).split()
    hyp_mots = normalisation_wer(hypothese).split()
    matcher  = difflib.SequenceMatcher(None, ref_mots, hyp_mots)
    erreurs  = []
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == 'replace':
            for r, h in zip(ref_mots[i1:i2], hyp_mots[j1:j2]):
                erreurs.append({'type':'substitution','ref':r,'hyp':h,'critique':r in MOTS_CLES_CRITIQUES})
        elif tag == 'delete':
            for r in ref_mots[i1:i2]:
                erreurs.append({'type':'suppression','ref':r,'hyp':'∅','critique':r in MOTS_CLES_CRITIQUES})
        elif tag == 'insert':
            for h in hyp_mots[j1:j2]:
                erreurs.append({'type':'insertion','ref':'∅','hyp':h,'critique':False})
    return erreurs

print("═"*65)
print("  ANALYSE DÉTAILLÉE DES ERREURS")
print("═"*65)

toutes_erreurs = []
for _, row in impact_df.iterrows():
    errs = analyser_erreurs(row['reference'], row['whisper_small'])
    if errs:
        print(f"\n  [Q{row['query_id']:>2}] WER={row['wer_small']:.3f}")
        print(f"    REF  : {row['reference']}")
        print(f"    SMALL: {row['whisper_small']}")
        for e in errs:
            crit = " ⚠️ CLÉ" if e['critique'] else ""
            print(f"    [{e['type']:<12}] {e['ref']!r:<18} → {e['hyp']!r}{crit}")
        for e in errs:
            e['query_id'] = row['query_id']
            toutes_erreurs.append(e)

errs_df = pd.DataFrame(toutes_erreurs) if toutes_erreurs else pd.DataFrame()
if len(errs_df) > 0:
    print(f"\n  Total erreurs small : {len(errs_df)}")
    print(f"  Erreurs sur mots-clés critiques : {errs_df['critique'].sum()}")
    print(f"  Répartition types : {errs_df['type'].value_counts().to_dict()}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 3.6 VISUALISATIONS PHASE 3
# ──────────────────────────────────────────────────────────────────────────────
if len(impact_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Graphe 1 : WER par requête
    ax = axes[0]
    x  = np.arange(len(impact_df))
    ax.bar(x-0.18, impact_df['wer_small'], 0.35, label='Whisper small',    color='#E07B54')
    ax.bar(x+0.18, impact_df['wer_large'], 0.35, label='Whisper large-v3', color='#3A86FF')
    ax.axhline(impact_df.wer_small.mean(), color='#E07B54', linestyle='--', lw=1.5)
    ax.axhline(impact_df.wer_large.mean(), color='#3A86FF', linestyle='--', lw=1.5)
    ax.set_xticks(x); ax.set_xticklabels([f'Q{i}' for i in impact_df.query_id], rotation=45, fontsize=8)
    ax.set_ylabel('WER'); ax.legend(fontsize=9)
    ax.set_title('WER par requête audio', fontweight='bold')
    ax.spines[['top','right']].set_visible(False)

    # Graphe 2 : Dégradation P@5
    ax = axes[1]
    ax.bar(x-0.25, impact_df['p5_ref'],   0.25, label='Référence',    color='#4CAF50')
    ax.bar(x,      impact_df['p5_small'], 0.25, label='Whisper small',    color='#E07B54')
    ax.bar(x+0.25, impact_df['p5_large'], 0.25, label='Whisper large-v3', color='#3A86FF')
    ax.set_xticks(x); ax.set_xticklabels([f'Q{i}' for i in impact_df.query_id], rotation=45, fontsize=8)
    ax.set_ylim(0, 1.15); ax.legend(fontsize=8)
    ax.set_title('P@5 : Référence vs Whisper small vs large', fontweight='bold')
    ax.spines[['top','right']].set_visible(False)

    plt.tight_layout()
    plt.savefig('phase3_resultats.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Sauvegardé : phase3_resultats.png")
else:
    print("⚠️  Pas de fichiers audio disponibles — graphes Phase 3 ignorés.")

---
# 🤖 PHASE 4 — Génération de Réponses avec LLM
*(15% de la note)*

**Modèle utilisé :** `mistralai/mixtral-8x7b-instruct-v0.1` via NVIDIA NIM — gratuit avec crédits offerts  
**Clé API :** https://build.nvidia.com → Get API Key (crédits gratuits)  


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 4.1 INITIALISATION DU CLIENT NVIDIA NIM
#
# NVIDIA NIM (Neural Inference Microservices) est une plateforme d'inférence
# qui expose une API 100% compatible avec le format OpenAI.
#
# Avantages :
#   - SDK standard OpenAI (pas de bibliothèque spéciale à installer)
#   - Modèle Mixtral 8x7B : excellent pour la génération de texte en français
#   - Crédits gratuits offerts à l'inscription (suffisant pour ce projet)
#   - Pas de rate-limit bloquant pour 50 requêtes
#
# Obtenir une clé : https://build.nvidia.com → Get API Key
# ──────────────────────────────────────────────────────────────────────────────

from openai import OpenAI

# Initialisation du client avec la base URL NVIDIA
# Le SDK OpenAI standard est réutilisé, seule la base_url change
client_nvidia = OpenAI(
    base_url=NVIDIA_BASE_URL,
    api_key=NVIDIA_API_KEY
)

# Modèle choisi : Mixtral 8x7B Instruct
# → Mixture-of-Experts : 8 experts de 7B params, activation de 2 à la fois
# → Très performant en français (entraîné sur des données multilingues)
# → Alternative : meta/llama-3.3-70b-instruct si Mixtral non disponible
MODELE_ACTIF       = 'mistralai/mixtral-8x7b-instruct-v0.1'
DELAI_ENTRE_APPELS = 1   # 1s entre appels (NVIDIA NIM a peu de restrictions)


def appeler_llm(prompt: str, retries: int = 3) -> str:
    """
    Appelle Mixtral 8x7B via NVIDIA NIM (format OpenAI-compatible).

    Architecture du message (format ChatML standard) :
      - system : définit le rôle et les contraintes du chatbot
      - user   : prompt contenant la requête client + les 3 produits

    Paramètres clés :
      - temperature=0.5 : équilibre créativité / cohérence
      - max_tokens=1024 : réponses suffisamment longues si besoin
      - top_p=1         : pas de nucleus sampling restrictif
      - stream=False    : on attend la réponse complète

    Paramètres :
        prompt  : texte de la requête utilisateur
        retries : nombre de tentatives en cas d'erreur (défaut 3)

    Retour :
        str : réponse du LLM ou message d'erreur formaté
    """
    for tentative in range(retries):
        try:
            completion = client_nvidia.chat.completions.create(
                model=MODELE_ACTIF,
                messages=[
                    {
                        'role': 'system',
                        'content': (
                            'Tu es un assistant e-commerce chaleureux pour Africouleur, '
                            'boutique spécialisée dans la mode et la décoration africaines '
                            '(bazin, wax, bogolan, bijoux...). '
                            'Tu réponds toujours en français, avec un ton amical et professionnel, '
                            'en 3 à 5 phrases maximum. '
                            'Tu ne dois jamais inventer de caractéristiques absentes des descriptions fournies.'
                        )
                    },
                    {'role': 'user', 'content': prompt}
                ],
                temperature=0.5,  # équilibre créativité et cohérence
                top_p=1,          # pas de nucleus sampling restrictif
                max_tokens=1024,  # largeur suffisante pour une bonne réponse
                stream=False      # on attend la réponse complète
            )
            # Extraction du texte de la réponse (format OpenAI standard)
            return completion.choices[0].message.content.strip()

        except Exception as e:
            msg = str(e)
            if '429' in msg or 'rate' in msg.lower():
                print(f'   ⏳ Rate limit — attente 30s (tentative {tentative+1}/{retries})...')
                time.sleep(30)
            else:
                print(f'   ⚠️  Erreur : {msg[:100]}')
                return f'[ERREUR: {msg[:80]}]'

    return '[ERREUR: nombre de tentatives épuisé]'


# Test de connexion
test_rep = appeler_llm('En une phrase, cite un tissu africain traditionnel.')
print(f'✅ API NVIDIA NIM opérationnelle')
print(f'   Modèle       : {MODELE_ACTIF}')
print(f'   Base URL     : {NVIDIA_BASE_URL}')
print(f'   Délai appels : {DELAI_ENTRE_APPELS}s')
print(f'   Test         : {test_rep}')
print(f'\n   ⏱️  Durée estimée — 50 requêtes  : ~{int(50*DELAI_ENTRE_APPELS/60)+2} min')
print(f'   ⏱️  Durée estimée — 3 variantes  : ~{int(30*DELAI_ENTRE_APPELS/60)+2} min')


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 4.2 LES 3 VARIANTES DE PROMPT
#
# Le sujet impose de tester et comparer 3 stratégies de prompting :
#
# V1 — Minimaliste
#   → Juste la requête + les produits, sans instructions de style
#   → Test du comportement par défaut du LLM
#
# V2 — Instructions détaillées
#   → Rôle précis, ton amical, format, longueur
#   → Réponses plus cohérentes et contrôlées
#
# V3 — Few-shot (exemples fournis)
#   → 2 exemples de bonnes réponses dans le prompt
#   → Le LLM s'aligne sur le format et le style attendus
# ──────────────────────────────────────────────────────────────────────────────

def formater_produits(top3: list) -> str:
    """Formate les 3 produits pour les insérer dans le prompt."""
    lignes = []
    for i, row in enumerate(top3, 1):
        nom   = str(row.get('name', ''))[:70]
        desc  = str(row.get('short_description', ''))[:120]
        prix  = str(row.get('price_fcfa', ''))
        lignes.append(f"{i}. {nom} — {desc} — {prix}")
    return '\n'.join(lignes)

def get_top3(requete: str) -> list:
    """Retourne les 3 produits les plus pertinents pour une requête."""
    resultats = recherche_semantique(requete, k=3)
    produits  = []
    for pid, _ in resultats:
        row = df[df['id'] == pid]
        if len(row) > 0:
            produits.append(row.iloc[0].to_dict())
    return produits


# ── VARIANTE 1 : Prompt minimaliste ──────────────────────────────────────────
def prompt_v1_minimal(requete: str, top3: list) -> str:
    """Prompt minimal : requête + produits, sans instructions de rôle ou de style."""
    return f"""Client : {requete}
Produits disponibles :
{formater_produits(top3)}
Réponse :"""


# ── VARIANTE 2 : Prompt avec instructions détaillées ─────────────────────────
def prompt_v2_detailed(requete: str, top3: list) -> str:
    """
    Prompt structuré avec rôle, ton, format et longueur spécifiés.
    Produit des réponses plus cohérentes et professionnelles.
    """
    return f"""Tu es un assistant e-commerce chaleureux pour Africouleur, boutique spécialisée
dans la mode et la décoration africaines (bazin, wax, bogolan, bijoux...).

Le client demande : "{requete}"

Produits les plus pertinents de notre catalogue :
{formater_produits(top3)}

Consignes :
- Répondre en français, ton amical et professionnel
- 3 à 5 phrases maximum
- Recommander le(s) produit(s) le(s) plus adapté(s) et expliquer pourquoi
- Ne pas inventer de caractéristiques absentes de la description

Réponse :"""


# ── VARIANTE 3 : Few-shot (2 exemples fournis) ────────────────────────────────
def prompt_v3_fewshot(requete: str, top3: list) -> str:
    """
    Prompt avec 2 exemples de bonne réponse (few-shot learning).
    Le LLM imite le style des exemples → réponses plus naturelles.
    """
    return f"""Tu es un assistant e-commerce pour la boutique Africouleur.

Voici des exemples de bonnes réponses :

Q: Je cherche un cadeau pour un mariage
R: Pour célébrer ce beau moment, je vous recommande chaleureusement notre Ensemble
Bayefall en bazin teint à la main. C'est une tenue 4 pièces absolument élégante à
157 430 FCFA, idéale pour les grandes occasions. Le bazin artisanal d'Afrique de l'Ouest
en fait un cadeau unique et mémorable !

Q: Avez-vous des bijoux en cauris ?
R: Bien sûr ! Notre Collier "Magie du cauris" est une pièce ethnique magnifique à 45€.
Le cauris est un symbole fort en Afrique de l'Ouest — ce bijou accompagnera aussi bien
une tenue traditionnelle que moderne. Nos boucles d'oreilles Peul fulani à 15€ sont
également un excellent choix pour compléter l'ensemble.

---
Maintenant, le client demande : "{requete}"

Produits pertinents :
{formater_produits(top3)}

Réponse :"""


print("✅ 3 variantes de prompt définies : v1 (minimal), v2 (détaillé), v3 (few-shot)")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 4.3 GÉNÉRATION SUR LES 50 REQUÊTES (prompt V2 — version robuste)
#
# Améliorations vs version précédente :
#   - Délai de 3s entre chaque appel (respecte la limite 30 req/min)
#   - Sauvegarde CSV après CHAQUE réponse (pas de perte si session expire)
#   - Reprise automatique : si le fichier CSV existe déjà, on saute
#     les requêtes déjà traitées et on repart de là où on s'est arrêté
# ──────────────────────────────────────────────────────────────────────────────

import os
FICHIER_GEN = 'resultats_phase4_generations.csv'

# ── Reprise automatique si fichier existant ───────────────────────────────────
# Utile si la session Colab a expiré en cours de route
if os.path.exists(FICHIER_GEN):
    gen_existant   = pd.read_csv(FICHIER_GEN)
    ids_deja_faits = set(gen_existant['query_id'].tolist())
    generations_50 = gen_existant.to_dict('records')
    print(f'♻️  Reprise détectée : {len(ids_deja_faits)}/50 requêtes déjà générées.')
    print(f'   → {50 - len(ids_deja_faits)} requête(s) restante(s).')
else:
    ids_deja_faits = set()
    generations_50 = []
    print('🆕 Démarrage génération complète (50 requêtes).')

restantes = [row for _, row in gt.iterrows() if int(row['ID']) not in ids_deja_faits]
print(f'\n⚙️  Génération en cours — modèle : {MODELE_ACTIF} | délai : {DELAI_ENTRE_APPELS}s/req')
print(f'   ⏱️  Durée estimée : ~{len(restantes)*DELAI_ENTRE_APPELS//60 + 1} min\n')

for row in tqdm(restantes, desc='LLM V2'):
    query = row['Requête client']
    top3  = get_top3(query)

    reponse = appeler_llm(prompt_v2_detailed(query, top3))

    # Pause respectant la limite RPM avant le prochain appel
    time.sleep(DELAI_ENTRE_APPELS)

    enr = {
        'query_id'   : int(row['ID']),
        'type'       : row['Type de requête'],
        'query'      : query,
        'top3_ids'   : str([p.get('id') for p in top3]),
        'top3_noms'  : ' | '.join(str(p.get('name',''))[:50] for p in top3),
        'response_v2': reponse,
    }
    generations_50.append(enr)

    # Sauvegarde progressive après chaque réponse
    pd.DataFrame(generations_50).to_csv(FICHIER_GEN, index=False)

gen_df = pd.read_csv(FICHIER_GEN)  # recharger depuis le CSV pour s'assurer de la cohérence
print(f'\n✅ {len(gen_df)} réponses générées et sauvegardées → {FICHIER_GEN}')
print()

# Aperçu de 3 exemples aléatoires
for _, r in gen_df.sample(min(3, len(gen_df)), random_state=42).iterrows():
    print(f'[Q{r["query_id"]}] {r["query"]}')
    print(f'→ {r["response_v2"][:180]}...')
    print()


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 4.4 COMPARAISON DES 3 VARIANTES DE PROMPT (sur 10 requêtes)
#
# Même approche robuste que la cellule 4.3 :
#   - 3 appels par requête (V1 + V2 + V3) → 30 appels au total
#   - Délai de 3s entre chaque appel
#   - Sauvegarde progressive + reprise automatique
#
# Ces 10 requêtes × 3 variantes seront notées par les 5 évaluateurs humains.
# ──────────────────────────────────────────────────────────────────────────────

FICHIER_3V     = 'resultats_phase4_comparaison3v.csv'
sample_queries = gt.sample(10, random_state=42).reset_index(drop=True)

# ── Reprise automatique ───────────────────────────────────────────────────────
if os.path.exists(FICHIER_3V):
    comp_existant = pd.read_csv(FICHIER_3V)
    ids_3v_faits  = set(comp_existant['query_id'].tolist())
    comparaison_3v = comp_existant.to_dict('records')
    print(f'♻️  Reprise 3V : {len(ids_3v_faits)}/10 requêtes déjà traitées.')
else:
    ids_3v_faits   = set()
    comparaison_3v = []
    print('🆕 Démarrage comparaison 3 variantes (10 requêtes × 3 prompts = 30 appels).')

restantes_3v = [row for _, row in sample_queries.iterrows() if int(row['ID']) not in ids_3v_faits]
print(f'   → {len(restantes_3v)} requête(s) restante(s) | ~{len(restantes_3v)*3*DELAI_ENTRE_APPELS//60 + 1} min\n')

for row in tqdm(restantes_3v, desc='3 variantes'):
    query = row['Requête client']
    top3  = get_top3(query)
    print(f'\n[Q{int(row["ID"]):>2}] {query[:55]}')

    # V1 — Prompt minimaliste
    r1 = appeler_llm(prompt_v1_minimal(query, top3))
    time.sleep(DELAI_ENTRE_APPELS)
    print(f'   V1 : {r1[:90]}...')

    # V2 — Instructions détaillées
    r2 = appeler_llm(prompt_v2_detailed(query, top3))
    time.sleep(DELAI_ENTRE_APPELS)
    print(f'   V2 : {r2[:90]}...')

    # V3 — Few-shot examples
    r3 = appeler_llm(prompt_v3_fewshot(query, top3))
    time.sleep(DELAI_ENTRE_APPELS)
    print(f'   V3 : {r3[:90]}...')

    enr = {
        'query_id'   : int(row['ID']),
        'type'       : row['Type de requête'],
        'query'      : query,
        'response_v1': r1,
        'response_v2': r2,
        'response_v3': r3,
    }
    comparaison_3v.append(enr)

    # Sauvegarde progressive
    pd.DataFrame(comparaison_3v).to_csv(FICHIER_3V, index=False)

comp_df = pd.read_csv(FICHIER_3V)
print(f'\n✅ Comparaison 3 variantes terminée : {len(comp_df)} requêtes → {FICHIER_3V}')


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 4.5 GRILLE D'ÉVALUATION HUMAINE
#
# Export d'un fichier CSV pour les 5 évaluateurs.
# Critères (1-5) : Pertinence / Naturalité / Utilité
#
# La grille couvre :
#   - Les 50 requêtes avec réponse V2 (évaluation principale)
#   - Les 10 requêtes avec 3 variantes (comparaison de prompting)
# ──────────────────────────────────────────────────────────────────────────────

# ── Grille principale (50 requêtes × V2) ────────────────────────────────────
eval_rows_50 = []
for evaluateur in range(1, 6):   # 5 évaluateurs
    for _, r in gen_df.iterrows():
        eval_rows_50.append({
            'Évaluateur'         : f'Évaluateur {evaluateur}',
            'Requête ID'         : r['query_id'],
            'Type'               : r['type'],
            'Requête client'     : r['query'],
            'Top-3 recommandés'  : ' | '.join(r['top3_noms']),
            'Réponse générée'    : r['response_v2'],
            'Pertinence (1-5)'   : '',
            'Naturalité (1-5)'   : '',
            'Utilité (1-5)'      : '',
            'Commentaire'        : '',
        })

grille_50_df = pd.DataFrame(eval_rows_50)
grille_50_df.to_csv('grille_evaluation_50req.csv', index=False)

# ── Grille comparaison 3 variantes (10 requêtes × 3 variantes) ──────────────
eval_rows_3v = []
for evaluateur in range(1, 6):
    for _, r in comp_df.iterrows():
        for var_label, var_col in [('V1 - Minimal','response_v1'),
                                    ('V2 - Détaillé','response_v2'),
                                    ('V3 - Few-shot','response_v3')]:
            eval_rows_3v.append({
                'Évaluateur'       : f'Évaluateur {evaluateur}',
                'Requête ID'       : r['query_id'],
                'Requête client'   : r['query'],
                'Variante'         : var_label,
                'Réponse générée'  : r[var_col],
                'Pertinence (1-5)' : '',
                'Naturalité (1-5)' : '',
                'Utilité (1-5)'    : '',
                'Commentaire'      : '',
            })

grille_3v_df = pd.DataFrame(eval_rows_3v)
grille_3v_df.to_csv('grille_evaluation_3variantes.csv', index=False)

print("✅ Grilles d'évaluation exportées :")
print(f"   📄 grille_evaluation_50req.csv      — {len(grille_50_df)} lignes ({len(gen_df)} requêtes × 5 évaluateurs)")
print(f"   📄 grille_evaluation_3variantes.csv — {len(grille_3v_df)} lignes (10 req × 3 variantes × 5 évaluateurs)")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 4.6 ANALYSE DES SCORES D'ÉVALUATION HUMAINE
#
# À exécuter APRÈS avoir collecté les notes des 5 évaluateurs.
# On calcule :
#   - Score moyen par critère
#   - Score moyen par variante de prompt
#   - Accord inter-annotateurs (alpha de Krippendorff)
# ──────────────────────────────────────────────────────────────────────────────

# !! Modifier le chemin ci-dessous si vous rechargez la grille remplie !!
GRILLE_REMPLIE_PATH = 'grille_evaluation_50req.csv'   # remplacer par la version remplie

try:
    grille_remplie = pd.read_csv(GRILLE_REMPLIE_PATH)

    # Conversion en numérique (les évaluateurs ont entré 1-5)
    for col in ['Pertinence (1-5)', 'Naturalité (1-5)', 'Utilité (1-5)']:
        grille_remplie[col] = pd.to_numeric(grille_remplie[col], errors='coerce')

    notes = grille_remplie.dropna(subset=['Pertinence (1-5)', 'Naturalité (1-5)', 'Utilité (1-5)'])

    if len(notes) > 0:
        print("═"*55)
        print("  SCORES D'ÉVALUATION HUMAINE")
        print("═"*55)
        for crit in ['Pertinence (1-5)', 'Naturalité (1-5)', 'Utilité (1-5)']:
            print(f"  {crit:<22} : {notes[crit].mean():.2f} / 5  (σ={notes[crit].std():.2f})")
        print(f"\n  Score global moyen : {notes[['Pertinence (1-5)','Naturalité (1-5)','Utilité (1-5)']].mean().mean():.2f} / 5")
        print("═"*55)

        # Accord inter-annotateurs simple (variance entre évaluateurs par requête)
        accord = notes.groupby('Requête ID')['Pertinence (1-5)'].std().mean()
        print(f"\n  Accord inter-annotateurs (écart-type moyen sur Pertinence) : {accord:.3f}")
        print(f"  (Valeur proche de 0 = bon accord entre évaluateurs)")
    else:
        print("⚠️  La grille n'est pas encore remplie. Distribuer aux évaluateurs.")

except FileNotFoundError:
    print("ℹ️  Grille non trouvée. À exécuter après avoir collecté les notes des évaluateurs.")
    print("   → Distribuer 'grille_evaluation_50req.csv' aux 5 évaluateurs")
    print("   → Collecter, fusionner, remplacer le fichier, relancer cette cellule")

---
# 🎁 BONUS — Pipeline Interactif Gradio
*(+10 points)*

Pipeline vocal complet : **Voix → ASR Whisper → Recherche sémantique → LLM → Réponse**

**Pourquoi Gradio plutôt que Streamlit ?**
- Fonctionne nativement dans Google Colab **sans ngrok**
- Génère un lien public `*.gradio.live` directement utilisable
- Interface prête en 1 cellule, pas de fichier externe
- Support natif de l'audio (micro ou fichier uploadé)

**Architecture de l'app :**
```
Onglet 1 — Requête TEXTE
  [Champ texte] → Recherche → Top-3 produits + Réponse LLM

Onglet 2 — Requête VOCALE
  [Fichier audio / micro] → Whisper → Texte → Recherche → Réponse LLM
```


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# BONUS — Installation Gradio
#
# gradio : framework d'interface web Python natif dans Colab
#   → Pas besoin de ngrok, pas de tunnel, lien public généré automatiquement
#   → share=True crée une URL gradio.live valable 72h
# ──────────────────────────────────────────────────────────────────────────────
!pip install -q gradio
import gradio as gr
print(f'✅ Gradio {gr.__version__} installé')


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# BONUS — APPLICATION GRADIO (pipeline complet)
#
# CORRECTIFS INTÉGRÉS :
#
# [FIX 1] Réponses trop courtes pour les articles multi-produits
#   → Prompt restructuré : liste explicite de chaque produit avec nom/desc/prix/lien
#   → max_tokens passé à 1024 (comme dans le notebook Phase 4)
#   → Suppression de la contrainte '3-5 phrases max' qui tronquait les réponses
#
# [FIX 2] Yoruba non transcrit (language='fr' forcé écrasait toutes les langues)
#   → Ajout d'un sélecteur de langue dans l'onglet Vocal
#   → Mode 'Auto-détection' : Whisper détecte la langue seul (supporte le yoruba)
#   → Mode 'Traduction → FR' : Whisper traduit directement yoruba/fon → français
#   → La transcription yoruba est traduite avant la recherche sémantique
# ──────────────────────────────────────────────────────────────────────────────

import gradio as gr
import tempfile, os


def formater_produits_html(top3: list) -> str:
    """
    Génère un tableau HTML des 3 produits trouvés.
    Affiché dans le composant gr.HTML() de l'interface.
    Inclut le lien vers le produit sur africouleur.com.
    """
    html = '<div style="font-family:sans-serif">'
    html += '<h4 style="color:#333;margin-bottom:10px">📦 Produits les plus pertinents</h4>'
    for i, p in enumerate(top3, 1):
        nom   = str(p.get('name',  'N/A'))[:80]
        desc  = str(p.get('short_description', ''))[:150]
        prix  = str(p.get('price_fcfa', 'N/A'))
        url   = str(p.get('url_product', '#'))
        html += f'''
        <div style="border:1px solid #e0e0e0;border-radius:8px;padding:12px;
                    margin-bottom:8px;background:#fafafa">
            <div style="font-weight:bold;color:#1a237e;font-size:14px">#{i} {nom}</div>
            <div style="color:#555;font-size:12px;margin:4px 0">{desc}</div>
            <div style="display:flex;justify-content:space-between;margin-top:6px;align-items:center">
                <span style="color:#2e7d32;font-weight:bold;font-size:13px">{prix}</span>
                <a href="{url}" target="_blank"
                   style="background:#1565c0;color:white;padding:4px 10px;
                          border-radius:4px;font-size:11px;text-decoration:none"
                >Voir le produit →</a>
            </div>
        </div>'''
    html += '</div>'
    return html


def pipeline_texte(requete: str):
    """
    Pipeline complet pour une requête texte.

    [FIX 1] Prompt revu : liste explicite de TOUS les produits avec détails complets.
    max_tokens=1024 pour ne pas tronquer les réponses multi-produits.
    """
    if not requete or not requete.strip():
        return '<p style="color:gray">Entrer une requête...</p>', ''

    # Étape 1 : recherche sémantique neuronale (MiniLM)
    top3_ids   = recherche_semantique(requete, k=3)
    top3_prods = []
    for pid, _ in top3_ids:
        row = df[df['id'] == pid]
        if len(row) > 0:
            top3_prods.append(row.iloc[0].to_dict())

    # Étape 2 : prompt structuré avec TOUS les détails produits
    # [FIX 1] Chaque produit est listé avec nom, description, prix ET lien
    # → le LLM ne peut plus ignorer les produits secondaires
    lignes_produits = '\n'.join([
        f"Produit {i+1} :\n"
        f"  - Nom         : {str(p.get('name',''))[:80]}\n"
        f"  - Description : {str(p.get('short_description',''))[:200]}\n"
        f"  - Prix        : {p.get('price_fcfa','N/A')}\n"
        f"  - Lien        : {p.get('url_product','#')}"
        for i, p in enumerate(top3_prods)
    ])

    prompt = (
        f'Le client demande : "{requete}"\n\n'
        f'Voici les {len(top3_prods)} produits les plus pertinents de notre catalogue :\n\n'
        f'{lignes_produits}\n\n'
        f'Instructions : Génère une réponse naturelle en français qui présente CHACUN des '
        f'{len(top3_prods)} produits listés ci-dessus avec leur lien. '
        f'Pour chaque produit, explique brièvement pourquoi il correspond à la demande du client. '
        f'Mentionne le prix en FCFA. Sois chaleureux et professionnel.'
    )

    # Étape 3 : appel LLM avec max_tokens=1024
    # [FIX 1] max_tokens aligné sur le notebook Phase 4 (cell 49)
    try:
        completion = client_nvidia.chat.completions.create(
            model=MODELE_ACTIF,
            messages=[
                {
                    'role': 'system',
                    'content': (
                        'Tu es un assistant e-commerce pour Africouleur, boutique spécialisée '
                        'dans la mode et la décoration africaines. '
                        'Tu dois TOUJOURS présenter TOUS les produits qui te sont fournis, '
                        'sans en omettre aucun. Réponds en français.'
                    )
                },
                {'role': 'user', 'content': prompt}
            ],
            temperature=0.5,
            top_p=1,
            max_tokens=1024,  # [FIX 1] 1024 au lieu de 400 → réponses complètes
            stream=False
        )
        reponse = completion.choices[0].message.content.strip()
    except Exception as e:
        reponse = f'[ERREUR LLM : {str(e)[:100]}]'

    return formater_produits_html(top3_prods), reponse


def pipeline_vocal(fichier_audio, mode_langue: str):
    """
    Pipeline vocal : audio → Whisper → (traduction si besoin) → recherche → LLM.

    [FIX 2] Gestion multilingue :
      - 'Français (fr)'       : transcription forcée en français
      - 'Auto-détection'      : Whisper détecte la langue seul (supporte le yoruba 'yo')
      - 'Yoruba (yo)'         : transcription yoruba puis traduction automatique → français
      - 'Traduction → FR'     : Whisper traduit directement depuis n'importe quelle langue

    Paramètres :
        fichier_audio : chemin du fichier audio (Gradio sauvegarde en tmp)
        mode_langue   : option sélectionnée dans l'interface
    """
    if fichier_audio is None:
        return 'Aucun fichier audio fourni.', '', ''

    # Charger le modèle ASR (réutilise whisper_small si déjà chargé)
    try:
        modele_asr = whisper_small
    except NameError:
        import whisper as _w
        modele_asr = _w.load_model('small')

    # [FIX 2] Paramètres Whisper selon le mode sélectionné
    if mode_langue == 'Français (fr)':
        # Transcription forcée en français
        result = modele_asr.transcribe(fichier_audio, language='fr', temperature=0)
        transcription = result['text'].strip()
        texte_recherche = transcription

    elif mode_langue == 'Yoruba (yo)':
        # Étape 1 : transcription en yoruba (language='yo')
        result_yo = modele_asr.transcribe(fichier_audio, language='yo',
                                          task='transcribe', temperature=0)
        transcription_yo = result_yo['text'].strip()

        # Étape 2 : traduction yoruba → français via Whisper (task='translate')
        # Whisper large-v3 fait la traduction directement sans modèle externe
        result_trad = modele_asr.transcribe(fichier_audio, language='yo',
                                            task='translate', temperature=0)
        transcription_fr = result_trad['text'].strip()

        # Affichage combiné yoruba + traduction
        transcription    = f'[Yoruba] {transcription_yo}\n[Français] {transcription_fr}'
        texte_recherche  = transcription_fr   # la recherche se fait sur le français

    elif mode_langue == 'Traduction → FR':
        # Whisper détecte la langue ET traduit directement vers le français
        # Parfait pour fon, wolof, haoussa et autres langues africaines
        result = modele_asr.transcribe(fichier_audio, task='translate', temperature=0)
        transcription   = result['text'].strip()
        texte_recherche = transcription

    else:  # Auto-détection (mode par défaut)
        # Whisper détecte automatiquement la langue, transcrit sans forçage
        # Supporte : fr, yo (yoruba), en, ar, sw (swahili) et 90+ autres langues
        result = modele_asr.transcribe(fichier_audio, temperature=0)
        langue_detectee = result.get('language', '?')
        transcription   = result['text'].strip()
        # Si la langue n'est pas le français, traduire pour la recherche
        if langue_detectee != 'fr':
            result_trad   = modele_asr.transcribe(fichier_audio, task='translate', temperature=0)
            texte_recherche = result_trad['text'].strip()
            transcription = f'[Langue détectée : {langue_detectee}] {transcription}\n[Traduit FR] {texte_recherche}'
        else:
            texte_recherche = transcription

    # Pipeline texte sur le texte en français
    produits_html, reponse = pipeline_texte(texte_recherche)

    return transcription, produits_html, reponse


# ── Construction de l'interface Gradio ───────────────────────────────────────
with gr.Blocks(
    title='Chatbot Africouleur',
    theme=gr.themes.Soft(primary_hue='blue'),
) as demo:

    gr.Markdown("""
    # 🛍️ Chatbot Africouleur
    **Assistant e-commerce multimodal** — Mode africaine & Décoration  
    *Pipeline : Whisper ASR → MiniLM multilingue → Mixtral 8x7B via NVIDIA NIM*
    """)

    # ── ONGLET 1 : Requête TEXTE ─────────────────────────────────────────────
    with gr.Tab('✍️  Requête texte'):
        gr.Markdown('Saisissez votre requête en français pour obtenir des recommandations personnalisées.')

        with gr.Row():
            texte_input = gr.Textbox(
                label='Votre requête',
                placeholder='Ex : Je cherche des vêtements pour mon bébé de 6 mois...',
                lines=2, scale=4
            )
            btn_texte = gr.Button('🔍 Rechercher', variant='primary', scale=1)

        gr.Examples(
            examples=[
                ['Je cherche un pagne wax pour une cérémonie'],
                ["Proposez-moi des articles pour un bébé de 8 mois"],
                ["J'ai besoin d'un cadeau original pour un mariage africain"],
                ['Avez-vous des bogolans du Mali ?'],
                ['Quels vêtements africains pour enfants avez-vous ?'],
            ],
            inputs=texte_input,
            label='Exemples de requêtes'
        )

        produits_html_1 = gr.HTML(label='Produits trouvés')
        reponse_llm_1   = gr.Textbox(
            label='💬 Réponse de l\'assistant (Mixtral 8x7B)',
            lines=8, interactive=False
        )
        btn_texte.click(
            fn=pipeline_texte,
            inputs=texte_input,
            outputs=[produits_html_1, reponse_llm_1]
        )

    # ── ONGLET 2 : Requête VOCALE ────────────────────────────────────────────
    with gr.Tab('🎙️  Requête vocale (FR / Yoruba / Auto)'):
        # [FIX 2] Explication du support multilingue
        gr.Markdown("""
        **Langues supportées :** Français · Yoruba · Auto-détection (Fon, Wolof, Haoussa...)  
        Sélectionnez le mode selon la langue de votre enregistrement.
        """)

        # [FIX 2] Sélecteur de langue
        mode_langue = gr.Radio(
            choices=[
                'Français (fr)',
                'Yoruba (yo)',
                'Traduction → FR',   # traduit directement depuis n'importe quelle langue
                'Auto-détection',    # Whisper détecte seul
            ],
            value='Auto-détection',  # mode par défaut le plus robuste
            label='Mode de transcription',
        )

        audio_input = gr.Audio(
            label='Fichier audio ou enregistrement micro (.m4a / .wav / .mp3)',
            type='filepath',
            sources=['upload', 'microphone']
        )
        btn_vocal = gr.Button('🎙️ Transcrire & Rechercher', variant='primary')

        # [FIX 2] La transcription affiche maintenant la langue détectée
        transcription_out = gr.Textbox(
            label='📝 Transcription Whisper (+ traduction si langue locale)',
            lines=3, interactive=False
        )
        produits_html_2 = gr.HTML(label='Produits trouvés')
        reponse_llm_2   = gr.Textbox(
            label='💬 Réponse de l\'assistant',
            lines=8, interactive=False
        )

        btn_vocal.click(
            fn=pipeline_vocal,
            inputs=[audio_input, mode_langue],   # [FIX 2] passage du mode_langue
            outputs=[transcription_out, produits_html_2, reponse_llm_2]
        )

    gr.Markdown("""
    ---
    *Projet NLP Avancé — Master ESGIS Bénin 2025-2026*  
    *Pipeline : Whisper large-v3 (ASR multilingue) + MiniLM multilingue (embeddings) + Mixtral 8x7B (NVIDIA NIM)*
    """)

print('✅ Interface Gradio construite avec correctifs FIX 1 et FIX 2')


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# BONUS — LANCEMENT DE L'APPLICATION GRADIO
#
# share=True  : génère un lien public gradio.live valable 72h
#               → Aucun compte ngrok requis, fonctionne nativement dans Colab
# debug=False : pas de logs verbeux dans l'interface
# quiet=True  : supprime les messages de démarrage Gradio
#
# Le lien public affiché (ex: https://abc123.gradio.live)
# peut être partagé pour la démo en soutenance.
# ──────────────────────────────────────────────────────────────────────────────

demo.launch(
    share=True,    # génère le lien public *.gradio.live
    debug=False,
    quiet=True
)


---
# 💾 EXPORT FINAL — Tous les résultats

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPORT DE TOUS LES FICHIERS DE RÉSULTATS
# ══════════════════════════════════════════════════════════════════════════════

# ── Phase 2 ──────────────────────────────────────────────────────────────────
eval_df[['query_id','type','query',
         'p5_tfidf','mrr_tfidf','hr5_tfidf',
         'p5_neural','mrr_neural','hr5_neural'
]].to_csv('resultats_phase2.csv', index=False)

# ── Phase 3 ──────────────────────────────────────────────────────────────────
if len(trans_df) > 0:
    trans_df.to_csv('resultats_phase3_wer.csv', index=False)
    impact_df.to_csv('resultats_phase3_impact.csv', index=False)

# ── Phase 4 ──────────────────────────────────────────────────────────────────
gen_df.to_csv('resultats_phase4_generations.csv', index=False)
comp_df.to_csv('resultats_phase4_comparaison3v.csv', index=False)

print("✅ Tous les fichiers de résultats exportés :")
print()
print("  PHASE 2 :")
print("  📄 resultats_phase2.csv                — métriques P@5/MRR/HR@5")
print("  🖼️  phase2_comparaison.png")
print("  🖼️  phase2_tsne.png")
print()
print("  PHASE 3 :")
print("  📄 resultats_phase3_wer.csv            — transcriptions + WER/CER")
print("  📄 resultats_phase3_impact.csv         — dégradation P@5 due à ASR")
print("  🖼️  phase3_resultats.png")
print()
print("  PHASE 4 :")
print("  📄 resultats_phase4_generations.csv    — 50 réponses LLM (V2)")
print("  📄 resultats_phase4_comparaison3v.csv  — 10 req × 3 variantes")
print("  📄 grille_evaluation_50req.csv         — grille vierge (5 évaluateurs)")
print("  📄 grille_evaluation_3variantes.csv    — grille comparaison prompts")

print()
print("═"*65)
print("  RÉCAPITULATIF GLOBAL DU PROJET")
print("═"*65)
print(f"  Catalogue          : {len(df)} produits scrapés sur africouleur.com")
print(f"  Requêtes évaluées  : {len(gt)} (15 directes + 20 besoin + 15 questions)")
print()
print(f"  [Phase 2] TF-IDF     P@5={eval_df.p5_tfidf.mean():.4f}  MRR={eval_df.mrr_tfidf.mean():.4f}  HR@5={eval_df.hr5_tfidf.mean():.4f}")
print(f"  [Phase 2] Neural     P@5={eval_df.p5_neural.mean():.4f}  MRR={eval_df.mrr_neural.mean():.4f}  HR@5={eval_df.hr5_neural.mean():.4f}")

if len(trans_df) > 0:
    print()
    print(f"  [Phase 3] WER moyen — Whisper small   : {trans_df.wer_small.mean():.4f}")
    print(f"  [Phase 3] WER moyen — Whisper large   : {trans_df.wer_large.mean():.4f}")
    print(f"  [Phase 3] Dégradation P@5 (small)     : {impact_df.degr_small.mean():+.4f}")

print()
print(f"  [Phase 4] {len(gen_df)} réponses LLM générées (Gemini 1.5 Flash)")
print(f"  [Phase 4] 3 variantes testées sur 10 requêtes")
print("═"*65)